# Sistema de conversión texto/glosas a video en LSEC

Este notebook permite reproducir el flujo:

**oración en español → glosas → video final en Lengua de Señas Ecuatoriana (LSEC)**

Para ejecutar el sistema se utilizan tres recursos:

1. `glosses.txt`: diccionario de correspondencias entre palabras/frases y glosas.
2. `sinonimos.txt`: archivo de sinónimos manuales.
3. `lenguaje_señas.zip`: carpeta comprimida con los videos de señas organizados por glosas y abecedario.

Los archivos `glosses.txt` y `sinonimos.txt` se descargan desde el repositorio de GitHub.  
La carpeta `lenguaje_señas` se descarga desde un archivo ZIP externo debido a su tamaño.

# Instalar dependencias

In [ ]:
# =========================================================
# 2. INSTALACIÓN DE DEPENDENCIAS
# =========================================================
# Esta celda instala las librerías necesarias para el procesamiento
# de texto, generación de glosas y manejo de video.

!pip install language-tool-python
!pip install nltk jellyfish
!pip install -U spacy spacy-curated-transformers
!python -m spacy download es_dep_news_trf
!python -c "import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')"
!pip install -U symspellpy
!wget -O es_50k.txt https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2018/es/es_50k.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_dep_news_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.

# Descargar el ZIP de lenguaje_señas

In [ ]:
# =========================================================
# 3. DESCARGA DE RECURSOS VISUALES
# =========================================================
# Esta celda descarga la carpeta comprimida lenguaje_señas.zip.
# El ZIP contiene los videos de señas organizados en carpetas
# de la 'a' a la 'z' y una carpeta adicional llamada abecedario.

import gdown
import zipfile
from pathlib import Path

PROJECT_DIR = Path("/content/lsec_demo")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = PROJECT_DIR / "lenguaje_senas.zip"

# Coloque aquí el ID del archivo compartido en Google Drive.
# Ejemplo de enlace:
# https://drive.google.com/file/d/ID_DEL_ARCHIVO/view?usp=sharing
FILE_ID = "1vtHqfekeVcY6-0t-LCXcC4VRKUg_65Yp"

if not ZIP_PATH.exists():
    print("Descargando lenguaje_senas.zip...")
    gdown.download(id=FILE_ID, output=str(ZIP_PATH), quiet=False)
else:
    print("El ZIP ya fue descargado:", ZIP_PATH)

print("Archivo ZIP:", ZIP_PATH)

Descargando lenguaje_senas.zip...


Downloading...
From (original): https://drive.google.com/uc?id=1vtHqfekeVcY6-0t-LCXcC4VRKUg_65Yp
From (redirected): https://drive.google.com/uc?id=1vtHqfekeVcY6-0t-LCXcC4VRKUg_65Yp&confirm=t&uuid=396e2089-d9e5-42df-917b-22561e233ad5
To: /content/lsec_demo/lenguaje_senas.zip
100%|██████████| 449M/449M [00:05<00:00, 87.9MB/s]

Archivo ZIP: /content/lsec_demo/lenguaje_senas.zip


# Extraer el ZIP

In [ ]:
# =========================================================
# 4. EXTRACCIÓN DE RECURSOS VISUALES
# =========================================================
# Esta celda extrae la carpeta lenguaje_señas dentro de /content/lsec_demo.
# La estructura esperada después de la extracción es:
#
# /content/lsec_demo/lenguaje_señas/
# ├── a/
# ├── b/
# ├── ...
# ├── z/
# └── abecedario/

SIGNS_ROOT = PROJECT_DIR / "lenguaje_señas"

if not SIGNS_ROOT.exists():
    print("Extrayendo recursos visuales...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(PROJECT_DIR)
else:
    print("La carpeta de señas ya fue extraída:", SIGNS_ROOT)

print("Carpeta de videos:", SIGNS_ROOT)

Extrayendo recursos visuales...
Carpeta de videos: /content/lsec_demo/lenguaje_señas


# Definir rutas reproducibles

In [ ]:
# =========================================================
# 5. CONFIGURACIÓN DE RUTAS DEL SISTEMA
# =========================================================
# Todas las rutas se definen de forma fija para evitar depender
# de rutas personales de Google Drive o cargas manuales de archivos.

from pathlib import Path

PROJECT_DIR = Path("/content/lsec_demo")
REPO_DIR = Path("/content/")

# Recursos lingüísticos desde GitHub
GLOSSES_PATH = REPO_DIR / "glosses.txt"
SYNONYMS_PATH = REPO_DIR / "sinonimos.txt"

# Recursos visuales desde el ZIP descargado
SIGNS_ROOT = PROJECT_DIR / "lenguaje_señas"
ALPHABET_ROOT = SIGNS_ROOT / "abecedario"

# Carpeta para guardar videos generados
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "video_lsec_final.mp4"

print("Diccionario de glosas:", GLOSSES_PATH)
print("Archivo de sinónimos:", SYNONYMS_PATH)
print("Carpeta de señas:", SIGNS_ROOT)
print("Carpeta de abecedario:", ALPHABET_ROOT)
print("Video de salida:", OUTPUT_VIDEO)

Diccionario de glosas: /content/glosses.txt
Archivo de sinónimos: /content/sinonimos.txt
Carpeta de señas: /content/lsec_demo/lenguaje_señas
Carpeta de abecedario: /content/lsec_demo/lenguaje_señas/abecedario
Video de salida: /content/lsec_demo/outputs/video_lsec_final.mp4


# Verificar recursos

In [ ]:
# =========================================================
# 6. VERIFICACIÓN DE RECURSOS
# =========================================================
# Esta celda comprueba que todos los archivos y carpetas necesarios
# existan antes de ejecutar el sistema.

required_paths = {
    "glosses.txt": GLOSSES_PATH,
    "sinonimos.txt": SYNONYMS_PATH,
    "carpeta lenguaje_señas": SIGNS_ROOT,
    "carpeta abecedario": ALPHABET_ROOT,
}

for name, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"No se encontró {name}: {path}")
    print(f"OK - {name}: {path}")

videos = list(SIGNS_ROOT.rglob("*.mp4"))

if len(videos) == 0:
    raise FileNotFoundError("No se encontraron videos .mp4 dentro de lenguaje_señas.")

print(f"Total de videos encontrados: {len(videos)}")

OK - glosses.txt: /content/glosses.txt
OK - sinonimos.txt: /content/sinonimos.txt
OK - carpeta lenguaje_señas: /content/lsec_demo/lenguaje_señas
OK - carpeta abecedario: /content/lsec_demo/lenguaje_señas/abecedario
Total de videos encontrados: 3484


In [ ]:
%run text_to_gloss.py

# PRUEBA DE CONVERSIÓN TEXTO → GLOSAS


In [ ]:
%run text_to_gloss.py

from text_to_gloss import TextToGlossTranslator
translator = TextToGlossTranslator(
        gloss_txt_path="glosses.txt",
        function_words=FUNCTION_WORDS,
        preserve_words=PRESERVE_WORDS,
        unknown_mode="spell",
        lt_language="es",
        spacy_model="es_dep_news_trf",
        disable_components=["ner"],
        manual_synonyms_txt_path="sinonimos.txt",
    )

oracion = "Mi hermano terminó su desayuno"
result = translator.translate(oracion)

print("=" * 120)
print("ENTRADA ORIGINAL   :", result["input_text"])
print("GLOSAS FINALES     :", " ".join(result["glosses"]))
print("MATCHES:")

translator.close()

ENTRADA ORIGINAL   : Mi hermano terminó su desayuno
GLOSAS FINALES     : MIO HERMANO TERMINAR SU DESAYUNO
MATCHES:


# Generar video desde glosas

In [ ]:
import sys
sys.path.append("/content/lsec_repo")  # cambia esto por la ruta real de tu repo clonado

from gloss_to_video import generar_video_desde_glosas

GLOSAS_FINALES = " ".join(result["glosses"])

video_final = generar_video_desde_glosas(
    glosas=GLOSAS_FINALES,
    signs_root="/content/lsec_demo/lenguaje_señas",
    output_path="/content/salida_final.mp4",
    show_plan=True,
    strict=True
)

print("Video generado:", video_final)


PLAN DE VIDEOS
[1] gloss=MIO
    unit = mio
    mode = sign
    found= True
    path = /content/lsec_demo/lenguaje_señas/m/Mío, a.mp4
[2] gloss=HERMANO
    unit = HERMANO
    mode = sign
    found= True
    path = /content/lsec_demo/lenguaje_señas/h/Hermano.mp4
[3] gloss=TERMINAR
    unit = TERMINAR
    mode = sign
    found= True
    path = /content/lsec_demo/lenguaje_señas/t/Acabar, Terminar.mp4
[4] gloss=SU
    unit = SU
    mode = sign
    found= True
    path = /content/lsec_demo/lenguaje_señas/s/Su, Suyo(a).mp4
[5] gloss=DESAYUNO
    unit = DESAYUNO
    mode = sign
    found= True
    path = /content/lsec_demo/lenguaje_señas/d/Desayuno.mp4

No faltan videos.
Cantidad de clips válidos: 5
1 /content/lsec_demo/lenguaje_señas/m/Mío, a.mp4
2 /content/lsec_demo/lenguaje_señas/h/Hermano.mp4
3 /content/lsec_demo/lenguaje_señas/t/Acabar, Terminar.mp4
4 /content/lsec_demo/lenguaje_señas/s/Su, Suyo(a).mp4
5 /content/lsec_demo/lenguaje_señas/d/Desayuno.mp4
OK: /content/lsec_demo/lenguaje_